# Experiment B2 — GPU steps-to-target vs batch size

This is the GPU/PyTorch version of `../cpu/CBS_steps.ipynb`.

The experiment fixes a target population excess loss and measures, for each batch
size $B$, the number of optimizer steps $K$ needed to first reach that target.
For every `(method, B)`, it tunes the step size $\eta$ and, for momentum methods,
the momentum coefficient $\rho$ on a small grid. The best configuration is the one
with the fewest average steps over seeds.

Methods:

- `SGD`
- `Heavy Ball (HB)`
- `Nesterov (NAG)`

The dashed reference line is $K \propto 1/B$.


In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

# ----------------------------- device -----------------------------
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float64
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = False
    print(f'Using GPU: {torch.cuda.get_device_name(DEVICE)}')
else:
    print('CUDA not available; falling back to CPU')
print(f'device={DEVICE}, dtype={DTYPE}')

# ----------------------------- copied setting from ../cpu/CBS_steps.ipynb -----------------------------
BETA = 2          # power-law spectrum exponent 'a' in the paper
SIGMA = 1.0
DIM = 4096        # coarse: 1024; paper-quality: 2048-4096
S_VALUE = 1       # source-condition smoothness

TARGET_LOSS = 0.006
K_MAX = 60_000
N_SEEDS = 3
CHUNK = 2048

ETA_GRID = [0.05, 0.1, 0.2, 0.4, 0.8]
ONE_MINUS_RHO_GRID = [0.2, 0.1, 0.05, 0.01, 0.005, 0.001]

B_EXPONENTS = list(range(1, 9))
B_LIST = [2 ** e for e in B_EXPONENTS]

ALGOS = {'HB': 'hb', 'Nesterov': 'nag', 'SGD': 'sgd'}
ALGO_SEED_OFFSET = {name: i for i, name in enumerate(ALGOS)}

LOAD_CACHE = True
SAVE_CACHE = True
SAVE_FIG = True
RESULT_DIR = Path('results')
RESULT_DIR.mkdir(exist_ok=True)

print(f'beta(a)={BETA}, sigma={SIGMA}, dim={DIM}, s={S_VALUE}')
print(f'target loss={TARGET_LOSS}, K_max={K_MAX}, seeds={N_SEEDS}')
print(f'batch grid: 2^{B_EXPONENTS[0]} .. 2^{B_EXPONENTS[-1]} ({len(B_LIST)} values)')
print(f'tuning grid: {len(ETA_GRID)} eta x {len(ONE_MINUS_RHO_GRID)} (1-rho) for momentum, {len(ETA_GRID)} eta for SGD')


In [ ]:
@torch.no_grad()
def build_problem(dim=DIM, beta=BETA, s=S_VALUE, device=DEVICE, dtype=DTYPE):
    """Return (lam, theta_star) for the diagonal linear-regression problem."""
    j = torch.arange(1, dim + 1, device=device, dtype=dtype)
    lam = j.pow(-beta)
    var_star = j.pow(-(1.0 + (s - 1.0) * beta))
    theta_star = torch.sqrt(var_star)
    return lam, theta_star


@torch.no_grad()
def excess_loss(delta, lam):
    """Exact population excess loss E = 1/2 * sum_j lam_j * delta_j^2."""
    return 0.5 * torch.sum(lam * delta * delta)


@torch.no_grad()
def grad_delta_chunked(delta, B, lam, sigma=SIGMA, generator=None, chunk=CHUNK):
    """Stochastic minibatch gradient without materializing a full (B, DIM) batch."""
    d = lam.numel()
    sqrt_lam = torch.sqrt(lam)
    grad = torch.zeros_like(delta)
    remaining = int(B)
    while remaining > 0:
        b = min(chunk, remaining)
        X = torch.randn((b, d), device=delta.device, dtype=delta.dtype, generator=generator)
        X.mul_(sqrt_lam)
        eps = torch.randn((b,), device=delta.device, dtype=delta.dtype, generator=generator)
        eps.mul_(sigma)
        resid = X.mv(delta).sub_(eps)
        grad.add_(X.t().mv(resid))
        remaining -= b
    grad.div_(float(B))
    return grad


@torch.no_grad()
def steps_to_target(algo, B, eta, rho, target, lam, theta_star,
                    sigma=SIGMA, K_max=K_MAX, seed=0):
    """Run one optimizer until target loss is reached; return (steps, final_loss)."""
    if algo == 'sgd':
        rho = 0.0
    generator = torch.Generator(device=lam.device)
    generator.manual_seed(int(seed))

    delta = -theta_star.clone()
    delta_prev = delta.clone()
    E = float(excess_loss(delta, lam).item())
    if E <= target:
        return 0, E

    for k in range(1, int(K_max) + 1):
        momentum = delta - delta_prev
        eval_delta = delta + rho * momentum if algo == 'nag' else delta
        g = grad_delta_chunked(eval_delta, B, lam, sigma=sigma, generator=generator)
        delta_new = delta - eta * g + rho * momentum
        delta_prev, delta = delta, delta_new
        E = float(excess_loss(delta, lam).item())
        if not np.isfinite(E) or E > 1e12:
            return int(K_max), float('inf')
        if E <= target:
            return k, E
    return int(K_max), E


def cache_key(prefix):
    device_tag = 'cuda' if DEVICE.type == 'cuda' else 'cpu'
    dtype_tag = str(DTYPE).split('.')[-1]
    tag = (
        f'{prefix}_gpu_torch_{device_tag}_{dtype_tag}_'
        f'a{BETA:g}_s{S_VALUE:g}_dim{DIM}_tgt{TARGET_LOSS:g}_'
        f'Kmax{K_MAX}_seeds{N_SEEDS}_B{B_EXPONENTS[0]}-{B_EXPONENTS[-1]}'
    ).replace('.', '_').replace('-', 'm')
    return RESULT_DIR / f'{tag}.npz'


lam, theta_star = build_problem()
E0 = float(excess_loss(-theta_star, lam).item())
print(f'initial excess loss E_0 = {E0:.4f}  -> target {TARGET_LOSS} is a {E0 / TARGET_LOSS:.0f}x reduction')


## Run the tuned steps-to-target sweep

For each $B$ and method, the notebook sweeps the tuning grid, averages steps over
seeds, and keeps the best configuration. A run that never reaches the target
counts as `K_MAX`, so divergent or under-tuned configurations are penalized.


In [ ]:
cache_path = cache_key('steps_to_target')

if LOAD_CACHE and cache_path.exists():
    RESULTS = np.load(cache_path, allow_pickle=True)['results'].item()
    print(f'loaded cached results from {cache_path}')
else:
    RESULTS = {'B': B_LIST}
    for name in ALGOS:
        RESULTS[name] = {'steps': [], 'reached': [], 'best_eta': [], 'best_rho': []}

    t0 = time.time()
    for B in B_LIST:
        print(f'B=2^{int(np.log2(B)):<2d} ({B:6d})')
        for name, algo in ALGOS.items():
            rho_grid = [0.0] if algo == 'sgd' else [1.0 - x for x in ONE_MINUS_RHO_GRID]
            best = {'steps': np.inf, 'eta': None, 'rho': None, 'reached': False}
            for eta in ETA_GRID:
                for rho in rho_grid:
                    step_vals = []
                    for seed in range(N_SEEDS):
                        run_seed = 2_000_000 + 1_000_000 * ALGO_SEED_OFFSET[name] + 100_000 * seed + B
                        k, _ = steps_to_target(algo, B, eta, rho, TARGET_LOSS,
                                               lam, theta_star, seed=run_seed)
                        step_vals.append(k)
                    mean_steps = float(np.mean(step_vals))
                    if mean_steps < best['steps']:
                        best = {'steps': mean_steps, 'eta': eta, 'rho': rho,
                                'reached': all(k < K_MAX for k in step_vals)}
            RESULTS[name]['steps'].append(best['steps'])
            RESULTS[name]['reached'].append(best['reached'])
            RESULTS[name]['best_eta'].append(best['eta'])
            RESULTS[name]['best_rho'].append(best['rho'])
            flag = '' if best['reached'] else '  [NOT reached: hit K_MAX]'
            print(f'  {name:9s} steps={best["steps"]:10.1f}  eta={best["eta"]}  rho={best["rho"]:.3f}{flag}')

        if SAVE_CACHE:
            np.savez_compressed(cache_path, results=RESULTS)

    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    print(f'finished in {(time.time() - t0) / 60:.2f} minutes')
    if SAVE_CACHE:
        np.savez_compressed(cache_path, results=RESULTS)
        print(f'saved results to {cache_path}')


## Plot: steps-to-target versus batch size

The dashed line is $K \propto 1/B$, anchored at the smallest-batch SGD point.
Hollow markers indicate runs that hit `K_MAX` before reaching the target; treat
those as lower bounds rather than exact measurements.


In [ ]:
def plot_steps_sweep(results):
    colors = {'HB': 'tab:orange', 'Nesterov': 'tab:purple', 'SGD': 'tab:green'}
    markers = {'HB': 's', 'Nesterov': 'o', 'SGD': '^'}

    B_arr = np.array(results['B'], dtype=float)
    fig, ax = plt.subplots(figsize=(7.0, 5.4))

    for name in ALGOS:
        steps = np.array(results[name]['steps'], dtype=float)
        reached = np.array(results[name]['reached'], dtype=bool)
        ax.plot(B_arr, steps, color=colors[name], label=name, zorder=2)
        ax.scatter(B_arr[reached], steps[reached], marker=markers[name],
                   color=colors[name], zorder=3)
        ax.scatter(B_arr[~reached], steps[~reached], marker=markers[name],
                   facecolors='none', edgecolors=colors[name], zorder=3)

    anchor_steps = float(results['SGD']['steps'][0])
    B0 = B_arr[0]
    ref = anchor_steps * (B0 / B_arr)
    ax.plot(B_arr, ref, 'k--', alpha=0.7, label=r'$1/B$ (iso-CE)', zorder=1)

    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    ax.set_xlabel('batch size  $B$')
    ax.set_ylabel('steps to target loss')
    ax.set_title(rf'GPU: $a={BETA:g}$, target loss ${TARGET_LOSS:g}$')
    ax.grid(True, which='both', color=(0, 0, 0, 0.15))
    ax.legend()
    plt.tight_layout()

    if SAVE_FIG:
        fig_path = RESULT_DIR / cache_key('steps_to_target').with_suffix('.pdf').name
        fig.savefig(fig_path, bbox_inches='tight')
        print(f'saved figure to {fig_path}')
    plt.show()

plot_steps_sweep(RESULTS)


## Notes

Expected behavior:

- All curves should roughly follow the $1/B$ line at small batch sizes, then flatten
  toward a serial-step floor. The elbow is the critical batch size.
- Nesterov may reach a lower serial floor than HB but peel off the linear-scaling
  window earlier.
- If many points are hollow, increase `K_MAX`, raise `TARGET_LOSS`, or widen the
  tuning grids.
- If momentum never wins, extend `ONE_MINUS_RHO_GRID` toward smaller $1-\rho$ and
  extend `ETA_GRID` upward.

GPU notes:

- The notebook uses `torch.float64` for closer agreement with the CPU notebook.
- On consumer GPUs, float64 can be much slower than float32; for a fast exploratory
  run you can change `DTYPE = torch.float32`.
